# Part 3 — Predictive Modeling with Spark MLlib
    **Prediction:** whether a customer subscribes to a term deposit (`y=yes`). This is a marketing response task, not loan-default prediction.

In [ ]:
# In Google Colab, uncomment if needed:
    # %pip -q install pyspark==3.5.3

In [ ]:
from pyspark.sql import SparkSession, functions as F
    from pyspark.ml import Pipeline
    from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
    from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
    from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
    import os, json
    spark = SparkSession.builder.appName("BankingSparkML").config("spark.sql.shuffle.partitions","4").getOrCreate()
    path = "../data/bank.csv"
    if not os.path.exists(path): path = "data/bank.csv"
    df = spark.read.option("header",True).option("inferSchema",True).csv(path)
    df = df.withColumn("label", F.when(F.col("y")=="yes",1.0).otherwise(0.0))
    df = df.withColumn("previously_contacted", F.when(F.col("pdays")==-1,"no").otherwise("yes"))
    df = df.dropna(subset=["label"])

## Leakage prevention
    The primary pre-contact model excludes `duration`: call duration is only known after the call and would leak information into a model intended to prioritize customers before contact. A retrospective model could include it, but it must be labeled separately.

In [ ]:
categorical = ["job","marital","education","default","housing","loan","contact","month","poutcome","previously_contacted"]
    numeric = ["age","balance","day","campaign","pdays","previous"]
    train, test = df.randomSplit([0.8,0.2], seed=42)
    indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in categorical]
    encoder = OneHotEncoder(inputCols=[c+"_idx" for c in categorical],
                            outputCols=[c+"_ohe" for c in categorical], handleInvalid="keep")
    imputer = Imputer(inputCols=numeric, outputCols=[c+"_imp" for c in numeric])
    assembler = VectorAssembler(inputCols=[c+"_imp" for c in numeric]+[c+"_ohe" for c in categorical],
                                outputCol="features", handleInvalid="keep")

In [ ]:
models = [
        ("logistic_regression", LogisticRegression(featuresCol="features", labelCol="label", maxIter=50, regParam=0.05)),
        ("random_forest", RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=80, maxDepth=8, seed=42))
    ]
    results = []
    for name, clf in models:
        pipe = Pipeline(stages=indexers+[encoder,imputer,assembler,clf])
        fitted = pipe.fit(train)
        pred = fitted.transform(test)
        roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC").evaluate(pred)
        pr = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR").evaluate(pred)
        f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1").evaluate(pred)
        results.append({"model":name,"roc_auc":roc,"pr_auc":pr,"weighted_f1":f1,
                        "train_rows":train.count(),"test_rows":test.count()})
        print(results[-1])

## Evaluation and business use
    Do not choose a model based on accuracy alone. The positive class is a minority, so examine PR-AUC and positive-class precision/recall alongside ROC-AUC. Weighted F1 may hide weak performance on the positive class. Only report the metrics printed by your actual run. For a stronger submission, add a confusion matrix and per-class precision/recall.

In [ ]:
spark.stop()